# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (attributes: name, description, etc.)
print(f"Dataset loaded: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant Record Sets are containers of logical tables or arrays. Here we enumerate all record sets and list their available fields and columns using their `@id` to maintain complete traceability.

In [ ]:
# List all record sets and their fields/columns, referencing by @id

from pprint import pprint

record_sets = dataset.metadata.record_sets
print(f"Available record sets:")
for rs in record_sets:
    print(f"- Record set @id: {rs.id}, name: {rs.name}")
    print("  Fields/columns in this record set:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print()


## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. Use the `@id` of record sets and fields as referenced above.

Below, we extract each record set by its `@id` and preview their columns and first few records.

In [ ]:
# Extract all record sets into DataFrames by @id

# Create a dictionary of @ids for all record sets
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2).to_string(index=False))
    else:
        print("  [No records found]")
    print()


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, we'll pick a representative numeric field from the main protein abundance record set, filter records, normalize a field, and group by an attribute. All variables below are referenced by their entity `@id` as per Croissant.

In [ ]:
# Choose a primary record set and numeric field for EDA

# Find the main protein table. You may need to replace these with the actual @id if your data structure differs.

selected_record_set_id = None
for rs in dataset.metadata.record_sets:
    # Look for a record set with 'protein' or a similar keyword in the name
    if 'protein' in rs.name.lower() or 'abundance' in rs.name.lower() or 'main' in rs.name.lower():
        selected_record_set_id = rs.id
        break
if selected_record_set_id is None and len(record_set_ids) > 0:
    selected_record_set_id = record_set_ids[0]  # fallback to first

print(f"Using record set: {selected_record_set_id}")

df = dataframes[selected_record_set_id]

# Choose a numeric field such as 'molecular_weight' (commonly MW) or 'abundance', referenced by @id
numeric_field_id = None
group_field_id = None
for field in dataset.metadata.record_set(selected_record_set_id).fields:
    field_name = field.name.lower()
    if ('weight' in field_name or 'abundance' in field_name or 'count' in field_name or 'coverage' in field_name or 'peptide' in field_name) and field.data_type in ['Float', 'Integer', 'Number']:
        numeric_field_id = field.id
        print(f"Selected numeric field @id: {numeric_field_id} (name: {field.name})")
        break
for field in dataset.metadata.record_set(selected_record_set_id).fields:
    if field.data_type == 'Text' and (field.name.lower().startswith('accession') or field.name.lower().startswith('category') or 'type' in field.name.lower()):
        group_field_id = field.id
        print(f"Selected group-by field @id: {group_field_id} (name: {field.name})")
        break

# If not found, list all possible fields for manual selection
if not numeric_field_id:
    print("Available numeric fields candidates:")
    pprint([(field.id, field.name, field.data_type) for field in dataset.metadata.record_set(selected_record_set_id).fields])

# Now perform the EDA if numeric_field_id is available
if numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records")
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Group mean by {group_field_id}:")
        print(grouped_df)
else:
    print(f"No numeric field detected in columns: {df.columns.tolist()}")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below displays a histogram of the selected numeric field, and a boxplot if grouped by a category field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of "{numeric_field_id}"')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} grouped by {group_field_id}')
        plt.xticks(rotation=90)
        plt.show()
else:
    print("No numeric field available for visualization.")


## 6. Conclusion
We successfully loaded the Mass Spectrometry EV dataset using `mlcroissant`, explored record sets and their fields using precise Croissant `@id` references, extracted and explored a main record set, and visualized key numeric variables.

**Next Steps:**
- Explore additional record sets and metadata fields as needed for downstream analysis.
- Apply advanced feature engineering, domain filtering, or modeling for scientific analysis.

For more details, consult the [Croissant documentation](https://mlcommons.org/croissant/specification/) or the dataset publisher's site.